# ABS Real Household Spending per Adult

How is the consumer travelling? Household spending per adult, in real terms, from two sources:

- **Monthly**: the Household Spending Indicator (5682.0, current prices, SA), deflated by a monthly SA headline CPI spliced from the monthly CPI (6401.0, from April 2024), the discontinued Monthly CPI Indicator (6484.0, September 2017 to September 2025) and the quarterly SA CPI interpolated to months before that.
- **Quarterly**: Household Final Consumption Expenditure from the National Accounts (5206.0, chain volume measures, SA).

**Population**: adults aged 21+. The labour force civilian population aged 15+ (6202.0) is the most up-to-date population measure, but its monthly path is stepped by the ABS benchmark interpolation. The level is rebuilt from the smoothed monthly increment, anchored at the latest published month, then scaled to the total resident population aged 21+ by the ERP 21+/15+ age share and the civilian-to-total upweight (3101.0, June benchmarks interpolated to months, held flat beyond the latest June).

**Deflator**: headline CPI is a broader basket than the Household Spending Indicator (it includes rents, for example), so the monthly deflator is an approximation.

**Benchmarking**: the Household Spending Indicator is benchmarked to the National Accounts. Over its eight categories that map cleanly to HFCE by purpose (all but Miscellaneous), its financial-year totals equal HFCE exactly (original series, current prices); the final chart shows the gap by financial year. This is inferred from the data, not from ABS documentation. Consequences:

- For benchmarked financial years the indicator carries no independent information on the *level* of spending; it distributes HFCE across months using its own transaction data. The cross-check chart is therefore partly comparing HFCE with itself.
- Growth comparisons that straddle financial years can be distorted by a shift in the within-year pattern. The 2019 divergence in the cross-check is such a case: the indicator front-loaded spending within FY2019 and FY2020, so its 2019 annual growth ran 3 to 4 points below HFCE. Prices (CPI vs the HFCE deflator) and scope (rent, energy, communications and education, which the indicator omits) do not explain it.
- The most recent months are the indicator's own, not yet benchmarked, reading, and will be revised when the National Accounts benchmark catches up.

## Python set-up

In [1]:
# system imports
from functools import cache

# analytic imports
import pandas as pd
import readabs as ra
from mgplot import bar_plot_finalise, line_plot_finalise, multi_start, series_growth_plot_finalise
from readabs import metacol as mc

# local imports
from abs_helper import get_abs_data
from abs_population import get_population, interp_21_share, interp_civ15_to_total
from abs_prices import get_cpi

In [2]:
# pandas display settings
pd.options.display.max_rows = 999999
pd.options.display.max_columns = 999

# display charts within this notebook
SHOW = False

# chart time ranges: full history and the last three years
plot_times = 0, -37
QUARTERLY_STARTS = 0, -13

# period and unit arithmetic
QUARTERS_PER_YEAR = 4
QTR_END_TO_MID = 1  # months from the last month of a quarter back to its middle month
PERCENT = 100
INDEX_BASE = 100
MILLION_PER_THOUSAND = 1_000  # $ Millions / '000 persons -> $ per person

# series selection
SA = "Seasonally Adjusted"
SPEND_TABLE = "5682001"
SPEND_DID = "Household spending ;  Total (Household Spending Categories) ;  Australia ;  Current Price ;"
CPI_SA_DID = "Index Numbers ;  All groups CPI, seasonally adjusted ;  Australia ;"
CPI_MONTHLY_TABLE = "640106"
CPI_INDICATOR_CAT = "6484.0"
CPI_INDICATOR_TABLE = "648401"
CPI_INDICATOR_URL = (
    "https://www.abs.gov.au/statistics/economy/price-indexes-and-inflation/"
    "monthly-consumer-price-index-indicator/latest-release"
)
HFCE_TABLE = "5206002_Expenditure_Volume_Measures"
HFCE_DID = "Households ;  Final consumption expenditure ;"

# benchmark check: Household Spending Indicator categories and their HFCE-by-purpose
# counterparts (Miscellaneous is left out: it covers only part of its HFCE match)
ORIGINAL = "Original"
SPEND_CATEGORY_TABLE = "5682002"
HFCE_PURPOSE_TABLE = "5206008_Household_Final_Consumption_Expenditure"
CATEGORY_MATCH = {
    "Food": ["Food"],
    "Alcoholic beverages and tobacco": ["Cigarettes and tobacco", "Alcoholic beverages"],
    "Clothing and footwear": ["Clothing and footwear"],
    "Furnishings and household equipment": ["Furnishings and household equipment"],
    "Health": ["Health"],
    "Transport": ["Purchase of vehicles", "Operation of vehicles", "Transport services"],
    "Recreation and culture": ["Recreation and culture"],
    "Hotels, cafes and restaurants": ["Hotels, cafes and restaurants"],
}

# chart footers
CPI_SOURCES = "6401.0, 6484.0"
POPULATION_SOURCES = "6202.0, 3101.0"
ADULTS = "Per adult aged 21+ (smoothed). "
MONTHLY_LFOOTER = f"Australia. {SA}. Deflated by monthly headline CPI (SA). {ADULTS}"
QUARTERLY_LFOOTER = f"Australia. {SA}. Chain volume measures. {ADULTS}"

## Data fetch

In [3]:
abs_dict, meta, source, _ = get_abs_data("5682.0", chart_dir_suffix=" - Real per Adult")

In [4]:
@cache
def get_nominal_spending() -> pd.Series:
    """Return monthly household spending, current prices, SA ($ Millions)."""
    selector = {SPEND_TABLE: mc.table, SPEND_DID: mc.did, SA: mc.stype}
    table, series_id, _units = ra.find_abs_id(meta, selector, verbose=False)
    series = abs_dict[table][series_id].dropna()
    if series.empty:
        raise ValueError("No monthly household spending data.")
    return series

In [5]:
@cache
def get_monthly_deflator() -> tuple[pd.Series, pd.DataFrame]:
    """Return a monthly SA headline CPI (latest month = 100) and its splice report.

    Segments, highest priority first: the monthly CPI (640106), the discontinued
    Monthly CPI Indicator (6484.0), and the quarterly SA CPI interpolated to months.
    rebase=True: this is an index spliced across reference-period changes.
    """
    selector = {CPI_SA_DID: mc.did, SA: mc.stype, "Month": mc.freq}
    current, current_meta = ra.read_abs_cat("6401.0", single_excel_only=CPI_MONTHLY_TABLE, verbose=False)
    indicator, indicator_meta = ra.read_abs_cat(
        CPI_INDICATOR_CAT, url=CPI_INDICATOR_URL, single_excel_only=CPI_INDICATOR_TABLE, verbose=False
    )
    monthly, discontinued = ra.select([(current, current_meta, selector), (indicator, indicator_meta, selector)])

    # qtly_to_monthly places each quarter on its last month; a quarterly index is
    # a quarter average, so it belongs on the middle month.
    quarterly, _units, _stype = get_cpi("headline_sa")
    interpolated = ra.qtly_to_monthly(quarterly, interpolate=True)
    interpolated.index = interpolated.index - QTR_END_TO_MID

    cpi, report = ra.splice([monthly, discontinued, interpolated], rebase=True)
    if cpi.empty or cpi.isna().any():
        raise ValueError("The spliced monthly CPI is empty or has gaps.")
    return cpi / cpi.iloc[-1] * INDEX_BASE, report

In [6]:
@cache
def get_adult_population() -> pd.Series:
    """Return the smoothed monthly population aged 21+ ('000).

    The civilian 15+ level is rebuilt from its smoothed monthly increment,
    anchored at the latest published month, then scaled to the total resident
    21+ population by the interpolated June age-share and civilian upweight.
    """
    increment, _units = get_population("civ15", smoothed=True)
    published, _units = get_population("civ15")
    published = published.reindex(increment.index)
    cumulative = increment.cumsum()
    smoothed = published.iloc[-1] - (cumulative.iloc[-1] - cumulative)
    gap = (smoothed / published - 1) * PERCENT
    print(f"Smoothed civilian 15+ level vs published: largest gap {gap.abs().max():.2f}%")

    months = smoothed.index
    if not isinstance(months, pd.PeriodIndex):
        raise TypeError("Expected a PeriodIndex on the civilian population.")
    adults = (smoothed * interp_21_share(months) * interp_civ15_to_total(months)).dropna()
    if adults.empty:
        raise ValueError("No adult population data.")
    return adults

In [7]:
@cache
def get_hfce() -> pd.Series:
    """Return quarterly Household Final Consumption Expenditure, CVM, SA ($ Millions)."""
    data, hfce_meta = ra.read_abs_cat("5206.0", single_excel_only=HFCE_TABLE, verbose=False)
    selector = {HFCE_TABLE: mc.table, HFCE_DID: mc.did, SA: mc.stype, "$ Millions": mc.unit}
    series = ra.select_one(data, hfce_meta, selector).dropna()
    if series.empty:
        raise ValueError("No household final consumption expenditure data.")
    return series

In [8]:
@cache
def get_benchmark_gap() -> pd.Series:
    """Return the financial-year gap (%) between the indicator and HFCE, matched categories.

    Both sides are Original, current prices, summed over CATEGORY_MATCH and over
    complete financial years (indexed by the year ending June). Zero means the
    indicator is benchmarked.
    """
    spend_selector = {SPEND_CATEGORY_TABLE: mc.table, ORIGINAL: mc.stype, "Month": mc.freq}
    spending = sum(
        ra.select_one(abs_dict, meta, spend_selector | {f"Household spending ;  {category} ;  Australia ;  Current Price ;": mc.did})
        for category in CATEGORY_MATCH
    )
    data, purpose_meta = ra.read_abs_cat("5206.0", single_excel_only=HFCE_PURPOSE_TABLE, verbose=False)
    hfce_selector = {HFCE_PURPOSE_TABLE: mc.table, ORIGINAL: mc.stype, "$ Millions": mc.unit}
    hfce = sum(
        ra.select_one(data, purpose_meta, hfce_selector | {f"{purpose}: Current prices ;": mc.did})
        for purposes in CATEGORY_MATCH.values()
        for purpose in purposes
    )
    if not isinstance(spending, pd.Series) or not isinstance(hfce, pd.Series):
        raise TypeError("Expected Series for the matched-category totals.")

    quarters = pd.DataFrame({"spending": ra.monthly_to_qtly(spending, f="sum"), "hfce": hfce}).dropna()
    if not isinstance(quarters.index, pd.PeriodIndex):
        raise TypeError("Expected a PeriodIndex on the quarterly totals.")
    financial_year = quarters.index.asfreq("Q-JUN").qyear
    totals = quarters.groupby(financial_year).sum()
    totals = totals[quarters.groupby(financial_year).size() == QUARTERS_PER_YEAR]
    if totals.empty:
        raise ValueError("No complete financial years for the benchmark check.")
    return (totals["spending"] / totals["hfce"] - 1) * PERCENT

## Real spending per adult

In [9]:
@cache
def get_real_spending_per_adult() -> pd.Series:
    """Return monthly real household spending per adult ($, latest-CPI-month prices)."""
    deflator, _report = get_monthly_deflator()
    real = get_nominal_spending() / deflator * INDEX_BASE
    return (real / get_adult_population() * MILLION_PER_THOUSAND).dropna()

In [10]:
@cache
def get_hfce_per_adult() -> pd.Series:
    """Return quarterly HFCE (CVM) per adult ($ per quarter)."""
    adults = ra.monthly_to_qtly(get_adult_population(), f="mean")
    return (get_hfce() / adults * MILLION_PER_THOUSAND).dropna()

In [11]:
def show_deflator_splice() -> None:
    """Print the audit report for the monthly CPI splice."""
    _cpi, report = get_monthly_deflator()
    print(report)


show_deflator_splice()

   segment         name freq_in  method  overlap_n window_start window_end  \
0        1  A128481587A       M  window         18      2024-04    2025-09   
1        2    A3604506F       M  window        105      2017-09    2026-05   

     factor fills_from  
0  0.809282    2017-09  
1  0.724033    1986-11  


## Plotting

In [12]:
def plot_monthly() -> None:
    """Plot monthly real household spending per adult: level and growth."""
    series = get_real_spending_per_adult()
    deflator, _report = get_monthly_deflator()
    prices = deflator.index[-1].strftime("%B %Y")
    common = {
        "starts": plot_times,
        "lfooter": MONTHLY_LFOOTER,
        "rfooter": f"{source}, {CPI_SOURCES}, {POPULATION_SOURCES}.",
        "show": SHOW,
    }
    multi_start(
        series,
        function=line_plot_finalise,
        title="Real household spending per adult",
        ylabel=f"$ per month ({prices} prices)",
        annotate=True,
        rounding=0,
        **common,
    )
    multi_start(
        series,
        function=series_growth_plot_finalise,
        title="Real household spending per adult: growth",
        y0=True,
        **common,
    )


plot_monthly()

Smoothed civilian 15+ level vs published: largest gap 0.22%


In [13]:
def plot_quarterly() -> None:
    """Plot quarterly HFCE per adult: level and growth."""
    series = get_hfce_per_adult()
    common = {
        "starts": QUARTERLY_STARTS,
        "lfooter": QUARTERLY_LFOOTER,
        "rfooter": f"ABS: 5206.0, {POPULATION_SOURCES}.",
        "show": SHOW,
    }
    multi_start(
        series,
        function=line_plot_finalise,
        title="Household consumption per adult",
        ylabel="$ per quarter (chain volume)",
        annotate=True,
        rounding=0,
        **common,
    )
    multi_start(
        series,
        function=series_growth_plot_finalise,
        title="Household consumption per adult: growth",
        y0=True,
        **common,
    )


plot_quarterly()

In [14]:
def plot_cross_check() -> None:
    """Compare annual growth in real spending per adult across the two sources."""
    monthly = ra.monthly_to_qtly(get_real_spending_per_adult(), f="sum")
    quarterly = get_hfce_per_adult()
    levels = pd.DataFrame(
        {
            "Household Spending Indicator, CPI deflated": monthly,
            "Household consumption (National Accounts), chain volume": quarterly,
        }
    )
    growth = ((levels / levels.shift(QUARTERS_PER_YEAR) - 1) * PERCENT).dropna()
    multi_start(
        growth,
        function=line_plot_finalise,
        starts=QUARTERLY_STARTS,
        title="Real spending per adult: annual growth by source",
        ylabel="Per cent",
        y0=True,
        annotate=True,
        legend=True,
        lfooter=f"Australia. {SA}. {ADULTS}",
        rfooter=f"{source}, 5206.0, {CPI_SOURCES}, {POPULATION_SOURCES}.",
        show=SHOW,
    )


plot_cross_check()

In [15]:
def plot_benchmark_gap() -> None:
    """Plot the financial-year gap between the indicator and HFCE (zero = benchmarked)."""
    bar_plot_finalise(
        get_benchmark_gap(),
        title="Household Spending Indicator vs HFCE: benchmark gap",
        xlabel="Financial year ending June",
        ylabel="Per cent",
        annotate=True,
        rounding=2,
        y0=True,
        lfooter=(
            f"Australia. {ORIGINAL}. Current prices. Eight matched spending categories. "
            "HFCE = Household Final Consumption Expenditure. "
        ),
        rfooter=f"{source}, 5206.0.",
        show=SHOW,
    )


plot_benchmark_gap()

## Watermark

In [16]:
# watermark
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-09-29 14:18:09

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.17.1

conda environment: n/a

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 27.0.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot : 0.3.0
pandas : 3.0.6
readabs: 0.3.0

Watermark: 2.6.0



In [17]:
print("Finished")

Finished
